# ODRD — coleta retrospectiva (busca de vídeos em alertas antigos)

**Por quê.** A cota da API ficou estourada em 45 de 60 dias, entre 25/08 e 13/10/2026. Nesse período, os alertas sem vídeo eram os últimos da fila, e até a v0.3 uma falha de cota era registrada como "nenhum vídeo encontrado". Os 449 alertas que saíram do painel sem vídeo estão contados como silêncio **sem prova de que a busca rodou**.

**O que este notebook faz**, com o mesmo método da v0.3.2 (código importado do repositório):
1. **Calibração** (38 alertas com vídeos conhecidos): mede quantos vídeos conhecidos a busca retrospectiva reencontra.
2. **Amostra** (120 de 449 alertas de silêncio, sistemática por UF × mês): mede quantos têm vídeo que cita o município.
3. **Restante** (opcional): os outros 329.

**Como ler o resultado.** A busca retrospectiva tende aos vídeos mais recentes e pode devolver zero em períodos de pouco interesse (Efstratiou, 2025):
- Achar um vídeo que cita o município **refuta** o silêncio daquele alerta.
- Não achar nada é **inconclusivo**.
- O resultado é um **limite inferior** dos falsos silêncios.

**Registro.** Este é um instrumento separado do pipeline: outro projeto do Google Cloud, outra chave, finalidade retrospectiva. Descreva-o assim no protocolo e no material do CEP. Os textos dos comentários ficam só no seu Drive; ao repositório vai apenas o `resultados_alertas.csv`, que tem contagens e IDs.

**Uso.**
- Ajuste os parâmetros e rode tudo (Ambiente de execução → Executar tudo).
- O notebook para sozinho ao atingir o limite de buscas e retoma de onde parou na execução seguinte. Rode uma vez por dia.
- A cota zera à meia-noite do Pacífico (4h ou 5h em Brasília).

In [ ]:
# ===== Parâmetros =====
FASE = "calibracao+amostra"     # "calibracao" | "amostra" | "calibracao+amostra" | "tudo"
MAX_BUSCAS_EXECUCAO = 95         # search.list: limite de 100/dia por projeto
MAX_UNIDADES_EXECUCAO = 9000     # demais chamadas: limite de 10.000/dia
COLETAR_COMENTARIOS = True       # comentários (janela de 72 h) dos vídeos que citam o município, só no grupo silêncio
MAX_PAGINAS_COMENTARIOS = 50     # 100 tópicos por página; a ordem é do mais novo ao mais antigo
PASTA = "/content/drive/MyDrive/ODRD/retro_backfill"
BRANCH = "main"                  # versão do código do pipeline usada (registrada nos resultados)

In [ ]:
# ===== Preparação =====
import csv, os, subprocess, sys, time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
REPO = "/content/crisis-dashboard"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/alvarocoutinho/crisis-dashboard.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from faa_poc import youtube
from faa_poc.config import CONFIG
from faa_poc.evidence import is_municipal, municipal_evidence
from faa_poc.geo import score_video
from faa_poc.timeutils import parse_dt

CFG = {**CONFIG, "MAX_VIDEOS_PER_ALERT": 10}   # igual ao pipeline (scripts/pipeline.py)
KEY = userdata.get("YT_key")
assert KEY, "Segredo 'YT_key' vazio: confira o ícone de chave na barra lateral e o acesso do notebook."

pasta = Path(PASTA); pasta.mkdir(parents=True, exist_ok=True)
ARQ_RES, ARQ_VID, ARQ_COM = pasta / "resultados_alertas.csv", pasta / "videos.csv", pasta / "comentarios.csv"
print(f"código do pipeline: {COMMIT} | saída: {pasta}")

In [ ]:
# ===== Universo e checkpoint =====
with open(f"{REPO}/data/retro/universo_alertas.csv", encoding="utf-8") as fh:
    universo = list(csv.DictReader(fh))

def _ler(arq):
    if not arq.exists():
        return []
    with open(arq, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

feitos = {r["alert_id"] for r in _ler(ARQ_RES) if r["status"] == "ok"}
filtro = {
    "calibracao": lambda r: r["grupo"] == "calibracao",
    "amostra": lambda r: r["amostra"] == "1",
    "calibracao+amostra": lambda r: r["grupo"] == "calibracao" or r["amostra"] == "1",
    "tudo": lambda r: True,
}[FASE]
pendentes = sorted((r for r in universo if filtro(r) and r["alert_id"] not in feitos), key=lambda r: int(r["ordem"]))
print(f"{len(universo)} alertas no universo; {len(feitos)} já coletados; {len(pendentes)} pendentes na fase '{FASE}'")
print(f"estimativa: ~{len(pendentes) * 2 / MAX_BUSCAS_EXECUCAO:.1f} execuções diárias (2 buscas por alerta)")

In [ ]:
# ===== Coleta =====
CAMPOS_RES = ["alert_id", "grupo", "amostra", "ordem", "municipio", "uf", "evento", "opened_at",
              "coletado_em", "codigo_pipeline", "busca_de", "busca_ate", "consultas", "status", "erro",
              "n_candidatos", "n_aceitos", "n_aceitos_municipais", "n_candidatos_municipais",
              "ids_aceitos", "ids_aceitos_municipais",
              "n_conhecidos", "n_conhecidos_nos_candidatos", "n_conhecidos_aceitos", "n_conhecidos_indisponiveis",
              "n_municipais_conhecidos", "n_municipais_conhecidos_reencontrados",
              "n_comentarios_janela", "status_comentarios"]
CAMPOS_VID = ["alert_id", "video_id", "aceito", "posicao", "search_query", "title", "channel_title", "published_at",
              "view_count", "comment_count", "relevance_score", "geo_score", "event_score", "geo_match_type",
              "municipal_evidence", "relevance_reasons", "rejection_reason"]
CAMPOS_COM = ["alert_id", "video_id", "comment_id", "parent_id", "is_reply", "published_at",
              "hours_from_alert_open", "like_count", "text_clean"]  # sem nome do autor (minimização)

def _anexar(arq, campos, linhas):
    novo = not arq.exists()
    with open(arq, "a", encoding="utf-8", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=campos, extrasaction="ignore")
        if novo:
            w.writeheader()
        w.writerows(linhas)

class Adiar(Exception):
    """Limite do dia atingido no meio do alerta: refaz na próxima execução."""

def coletar(row, quota):
    alert = {"cod_alerta": row["cod_alerta"], "municipio": row["municipio"], "uf": row["uf"],
             "codibge": row["codibge"], "evento": row["evento"], "nivel": row["nivel"],
             "datahoracriacao": row["opened_at"]}
    after, before, _ = youtube.search_window(alert, CFG)
    queries = youtube.build_queries(alert)
    res = {k: row.get(k, "") for k in ("alert_id", "grupo", "amostra", "ordem", "municipio", "uf", "evento", "opened_at")}
    res.update(coletado_em=datetime.now(timezone.utc).isoformat(timespec="seconds"), codigo_pipeline=COMMIT,
               busca_de=after, busca_ate=before, consultas=" | ".join(queries), status="ok", erro="")

    # 1. Busca: as mesmas 2 consultas do pipeline, na janela do alerta
    seen = {}
    for q in queries:
        if not quota.can_search(True):
            raise Adiar()
        r = youtube.search(q, after, before, KEY, CFG["MAX_VIDEOS_PER_ALERT"])
        quota.charge_search()
        if r.status == "quota_exhausted":
            quota.search_exhausted = True
            raise Adiar()
        if r.status != "ok":
            res.update(status="api_error", erro=r.error[:200])
            return res, [], []
        for v in r.items:
            seen.setdefault(v["video_id"], {**v, "search_query": q})
    cands = list(seen.values())

    # 2. Estatísticas + descrição completa (1 unidade para até 50 vídeos)
    stats = {}
    if cands:
        if not quota.can_spend(1):
            raise Adiar()
        s = youtube.stats([v["video_id"] for v in cands], KEY); quota.charge(1)
        if s.status == "quota_exhausted":
            quota.exhausted = True
            raise Adiar()
        stats = {x["video_id"]: x for x in s.items}

    # 3. Filtro de relevância + evidência municipal (v0.3.2), mesma ordenação do pipeline
    scored = []
    for v in cands:
        sc = score_video(v, alert, CFG)
        ev = municipal_evidence(row["municipio"], row["uf"], title=v.get("title", ""),
                                description=stats.get(v["video_id"], {}).get("description", "") or v.get("description_snippet", ""))
        scored.append((v, sc, ev))
    aceitos = [t for t in scored if t[1]["is_relevant"]]
    aceitos.sort(key=lambda t: (is_municipal(t[2]), t[1]["geo_match_type"] == "municipality_direct", t[1]["relevance_score"],
                                int(stats.get(t[0]["video_id"], {}).get("comment_count", 0) or 0),
                                int(stats.get(t[0]["video_id"], {}).get("view_count", 0) or 0)), reverse=True)
    aceitos = aceitos[:CFG["MAX_VIDEOS_PER_ALERT"]]
    ids_aceitos = [t[0]["video_id"] for t in aceitos]
    ids_mun = [t[0]["video_id"] for t in aceitos if is_municipal(t[2])]

    videos = []
    for v, sc, ev in scored:
        st = stats.get(v["video_id"], {})
        videos.append({"alert_id": row["alert_id"], "video_id": v["video_id"], "aceito": int(v["video_id"] in ids_aceitos),
                       "posicao": ids_aceitos.index(v["video_id"]) + 1 if v["video_id"] in ids_aceitos else "",
                       "search_query": v["search_query"], "title": v.get("title", ""), "channel_title": v.get("channel_title", ""),
                       "published_at": v.get("published_at", ""), "view_count": st.get("view_count", ""),
                       "comment_count": st.get("comment_count", ""), "municipal_evidence": ev,
                       **{k: sc[k] for k in ("relevance_score", "geo_score", "event_score", "geo_match_type",
                                             "relevance_reasons", "rejection_reason")}})
    res.update(n_candidatos=len(cands), n_aceitos=len(ids_aceitos), n_aceitos_municipais=len(ids_mun),
               n_candidatos_municipais=sum(1 for _, _, ev in scored if is_municipal(ev)),
               ids_aceitos=";".join(ids_aceitos), ids_aceitos_municipais=";".join(ids_mun))

    # 4. Calibração: quantos vídeos conhecidos a busca reencontrou?
    conhecidos = [v for v in row["videos_conhecidos"].split(";") if v]
    mun_conhecidos = [v for v in row["videos_municipais_conhecidos"].split(";") if v]
    if conhecidos:
        faltam = [v for v in conhecidos if v not in seen]
        indisponiveis = 0
        if faltam and quota.can_spend(1):   # sumiu do YouTube ou a busca não devolveu?
            chk = youtube.stats(faltam, KEY); quota.charge(1)
            if chk.status == "ok":
                indisponiveis = len(set(faltam) - {x["video_id"] for x in chk.items})
        res.update(n_conhecidos=len(conhecidos), n_conhecidos_nos_candidatos=len(conhecidos) - len(faltam),
                   n_conhecidos_aceitos=sum(1 for v in conhecidos if v in ids_aceitos),
                   n_conhecidos_indisponiveis=indisponiveis, n_municipais_conhecidos=len(mun_conhecidos),
                   n_municipais_conhecidos_reencontrados=sum(1 for v in mun_conhecidos if v in seen))

    # 5. Comentários da janela de 72 h, só de vídeos que citam o município (grupo silêncio)
    comentarios = []
    if COLETAR_COMENTARIOS and row["grupo"] == "silencio" and ids_mun:
        opened = parse_dt(row["opened_at"])
        fim = opened + timedelta(hours=CFG["EVENT_WINDOW_HOURS"]) if opened else None
        status_c = "ok"
        for vid in ids_mun:
            c = youtube.comments(vid, KEY, max_pages=MAX_PAGINAS_COMENTARIOS, order="time", quota=quota, window_end=fim)
            if c.status != "ok":
                status_c = c.status
            canon, links, _ = youtube.comment_links(c.items, alert=alert, link_scope="municipality_direct",
                                                     window_hours=CFG["EVENT_WINDOW_HOURS"])
            horas = {l["comment_id"]: l["hours_from_alert_open"] for l in links}
            for x in canon:
                comentarios.append({"alert_id": row["alert_id"], **{k: x.get(k, "") for k in CAMPOS_COM if k in x},
                                    "hours_from_alert_open": horas.get(x["comment_id"], "")})
        res.update(n_comentarios_janela=len(comentarios), status_comentarios=status_c)
    return res, videos, comentarios

quota = youtube.QuotaTracker(ceiling=MAX_UNIDADES_EXECUCAO, high_reserve=0,
                             search_ceiling=MAX_BUSCAS_EXECUCAO, search_high_reserve=0)
n_ok = 0
for row in pendentes:
    try:
        res, videos, comentarios = coletar(row, quota)
    except Adiar:
        print(f"limite do dia atingido em {row['alert_id']}: rode de novo amanhã (retoma daqui)")
        break
    _anexar(ARQ_VID, CAMPOS_VID, videos)
    _anexar(ARQ_COM, CAMPOS_COM, comentarios)
    _anexar(ARQ_RES, CAMPOS_RES, [res])   # por último: só conta como feito se tudo foi gravado
    n_ok += 1
    print(f"{row['ordem']:>4} {row['alert_id']:18s} {row['municipio'][:22]:22s} {row['grupo']:10s} "
          f"cand {res.get('n_candidatos', 0):>2} aceitos {res.get('n_aceitos', 0):>2} "
          f"citam município {res.get('n_aceitos_municipais', 0):>2} {res['status']}")
    time.sleep(0.2)
print(f"\n{n_ok} alertas nesta execução | buscas {quota.searches}/{MAX_BUSCAS_EXECUCAO} | unidades {quota.used}/{MAX_UNIDADES_EXECUCAO}")

In [ ]:
# ===== Resumo =====
import math
from collections import Counter

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"),) * 3
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

res = [r for r in _ler(ARQ_RES) if r["status"] == "ok"]
cal = [r for r in res if r["grupo"] == "calibracao"]
if cal:
    tot = sum(int(r["n_conhecidos"] or 0) for r in cal)
    achou = sum(int(r["n_conhecidos_nos_candidatos"] or 0) for r in cal)
    sumiu = sum(int(r["n_conhecidos_indisponiveis"] or 0) for r in cal)
    mt = sum(int(r["n_municipais_conhecidos"] or 0) for r in cal)
    ma = sum(int(r["n_municipais_conhecidos_reencontrados"] or 0) for r in cal)
    al = sum(1 for r in cal if int(r["n_municipais_conhecidos"] or 0) > 0)
    al_ok = sum(1 for r in cal if int(r["n_municipais_conhecidos"] or 0) > 0 and int(r["n_aceitos_municipais"] or 0) > 0)
    p, lo, hi = wilson(al_ok, al)
    print(f"CALIBRAÇÃO ({len(cal)} alertas)")
    print(f"  vídeos conhecidos reencontrados: {achou}/{tot} ({achou / max(tot, 1):.0%}); indisponíveis hoje: {sumiu}")
    print(f"  vídeos municipais conhecidos reencontrados: {ma}/{mt} ({ma / max(mt, 1):.0%})")
    print(f"  SENSIBILIDADE por alerta (tinha vídeo municipal → a busca acha ≥1): {al_ok}/{al} = {p:.0%} [IC95% {lo:.0%}–{hi:.0%}]")

am = [r for r in res if r["grupo"] == "silencio" and r["amostra"] == "1"]
if am:
    k = sum(1 for r in am if int(r["n_aceitos_municipais"] or 0) > 0)
    kv = sum(1 for r in am if int(r["n_aceitos"] or 0) > 0)
    p, lo, hi = wilson(k, len(am))
    print(f"\nAMOSTRA DE SILÊNCIO ({len(am)} de 120 alertas)")
    print(f"  silêncio REFUTADO (≥1 vídeo aceito que cita o município): {k}/{len(am)} = {p:.0%} [IC95% {lo:.0%}–{hi:.0%}]")
    print(f"  extrapolado para 449: ~{p * 449:.0f} [{lo * 449:.0f}–{hi * 449:.0f}] alertas (limite inferior)")
    print(f"  com algum vídeo aceito (inclui regional, H1): {kv}/{len(am)}")
    por_uf = Counter(r["uf"] for r in am if int(r["n_aceitos_municipais"] or 0) > 0)
    print(f"  refutados por UF: {dict(por_uf)}")
    if cal and al:
        print("  Leitura: se a sensibilidade da calibração for baixa, a taxa refutada subestima os falsos silêncios.")

## Depois da coleta

1. Baixe `resultados_alertas.csv` da pasta do Drive. Ele tem só contagens e IDs, sem texto.
2. Mande o arquivo para o Claude, ou rode no repositório:
   `python scripts/merge_retro_backfill.py --resultados caminho/resultados_alertas.csv`
3. `videos.csv` e `comentarios.csv` **ficam no Drive**: títulos e textos de comentários não vão para o repositório público.

**Referência.** Efstratiou, A. (2025). *I'm sorry Dave, I'm afraid I can't return that: On YouTube Search API use in research* (arXiv:2506.04422). https://arxiv.org/abs/2506.04422